# SAFE-VISION: Facial Feature-Based Child Age Estimation (CPU-Compatible Pipeline)

This notebook implements a complete pipeline to detect faces and estimate whether a user is a **Child** or **Not a Child** (Adult/Teen) using geometric facial features, OpenCV, and Scikit-Learn classification models. 

To ensure compatibility across systems lacking AVX/AVX2 instruction sets (preventing PyTorch/TensorFlow DLL errors), this pipeline relies on native CPU-optimized machine learning models (`RandomForestClassifier` and `MLPClassifier` neural networks).

### Pipeline Stages:
1. **Part 1: Face Detection**: Uses OpenCV Haar Cascades to locate and crop faces from input media.
2. **Part 2: Geometric & Visual Feature Extraction**: Computes roundness, aspect ratio, and vertical facial intensity projections (eye and mouth relative layout).
3. **Part 3: Scikit-Learn Classifiers**: Sets up Random Forest and Multi-Layer Perceptron (MLP) age classification models.
4. **Part 4: Synthetic Data & Training**: Generates synthetic faces (Child round faces vs. Adult long faces) to train and evaluate the classifiers.
5. **Part 5: Inference & Visualization**: Takes a new face, extracts features, predicts age group, and plots the classification.

## Part 1: Face Detection and Preprocessing

We load Haar Cascades built into OpenCV to locate faces and crop them to a normalized 128x128 resolution.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

# Load built-in Haar Cascade classifier from OpenCV
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_and_crop_face(image_np: np.ndarray) -> tuple:
    """
    Detects the primary face in an image, crops it, and returns the cropped face and bounding box.
    """
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    
    if len(faces) == 0:
        # If no face is detected, return original image (scaled) and None
        return cv2.resize(image_np, (128, 128)), None
        
    # Select the largest face detected
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    cropped_face = image_np[y:y+h, x:x+w]
    cropped_face_resized = cv2.resize(cropped_face, (128, 128))
    
    return cropped_face_resized, (x, y, w, h)

## Part 2: Geometric & Visual Feature Extraction

We extract geometric features (facial aspect ratio, roundness) and visual features (using vertical pixel density projections to capture the vertical spacing of eyes and mouth).

In [ ]:
def extract_features(cropped_face: np.ndarray, bbox: tuple) -> np.ndarray:
    """
    Extracts a feature vector for classification:
    - Aspect ratio & face roundness (2 features)
    - Vertical projection (eye-to-mouth ratio) of cropped face (10 downsampled bin features)
    """
    # 1. Bounding box features
    if bbox is not None:
        x, y, w, h = bbox
        aspect_ratio = w / h
        roundness = min(w, h) / max(w, h)
    else:
        aspect_ratio = 1.0
        roundness = 1.0
        
    # 2. Vertical intensity projections (helps detect vertical spacing differences)
    gray_face = cv2.cvtColor(cropped_face, cv2.COLOR_RGB2GRAY)
    vertical_projection = np.mean(gray_face, axis=1)  # Mean intensity per row
    
    # Downsample projection to 10 bin values to prevent overfitting
    projection_bins = np.array([np.mean(chunk) for chunk in np.array_split(vertical_projection, 10)])
    projection_bins_normalized = (projection_bins - np.mean(projection_bins)) / (np.std(projection_bins) + 1e-6)
    
    # Concatenate features into a single 12-dimensional vector
    feature_vector = np.concatenate(([aspect_ratio, roundness], projection_bins_normalized))
    return feature_vector

## Part 3: Age Classifier using Scikit-Learn

We load Random Forest and Multi-Layer Perceptron (MLP) classifiers to train on our extracted features.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize models
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
mlp_model = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)

## Part 4: Synthetic Dataset & Training

We write a helper to generate synthetic faces. 
- **Child faces**: Round profile shape with eyes positioned lower and larger circles.
- **Adult faces**: Elongated face profile, smaller eyes positioned higher.

In [ ]:
def generate_synthetic_face(is_child: bool) -> np.ndarray:
    """
    Generates a synthetic geometric face image representing age features.
    """
    img = np.ones((128, 128, 3), dtype=np.uint8) * 240  # Light background
    cx, cy = 64, 64
    
    if is_child:
        # Round face shape
        rx, ry = random.randint(40, 48), random.randint(40, 48)
        cv2.ellipse(img, (cx, cy), (rx, ry), 0, 0, 360, (255, 200, 180), -1)  # Skin tone
        # Lower eyes, larger circles
        eye_y = cy + 5
        eye_r = random.randint(6, 8)
    else:
        # Oval, elongated face shape
        rx, ry = random.randint(34, 40), random.randint(50, 56)
        cv2.ellipse(img, (cx, cy), (rx, ry), 0, 0, 360, (245, 190, 160), -1)
        # Higher eyes, smaller circles
        eye_y = cy - 10
        eye_r = random.randint(4, 5)
        
    # Draw eyes
    cv2.circle(img, (cx - 15, eye_y), eye_r, (40, 40, 40), -1)
    cv2.circle(img, (cx + 15, eye_y), eye_r, (40, 40, 40), -1)
    
    # Draw mouth
    cv2.ellipse(img, (cx, cy + 20), (12, 6), 0, 0, 180, (200, 50, 50), 2)
    
    # Add noise to simulate real-world frame variability
    noise = np.random.normal(0, 5, img.shape).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return img

# Compile datasets
num_samples = 300
features_list = []
labels_list = []

for _ in range(num_samples):
    label = random.choice([0, 1])  # 0 -> Child, 1 -> Not a Child
    face_img = generate_synthetic_face(is_child=(label == 0))
    
    # Run detection and extract features
    cropped, bbox = detect_and_crop_face(face_img)
    feats = extract_features(cropped, bbox)
    
    features_list.append(feats)
    labels_list.append(label)

X = np.array(features_list)
y = np.array(labels_list)

# Split into Train and Validation sets
split_idx = int(0.8 * num_samples)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"Generated {num_samples} face samples.")
print(f"Feature dimensions: {X.shape} | Labels dimensions: {y.shape}")

### Training and Evaluating the Models

In [ ]:
# Train Random Forest
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_val)
rf_acc = accuracy_score(y_val, rf_preds)

# Train MLP Classifier (Neural Network)
mlp_model.fit(X_train, y_train)
mlp_preds = mlp_model.predict(X_val)
mlp_acc = accuracy_score(y_val, mlp_preds)

print("=" * 40)
print(f"Random Forest Validation Accuracy : {rf_acc:.2%}")
print(f"MLP Classifier Validation Accuracy: {mlp_acc:.2%}")
print("=" * 40)
print("\nMLP Classification Report:")
print(classification_report(y_val, mlp_preds, target_names=["Child", "Not a Child"]))

## Part 5: Inference and Visualization Pipeline

This function receives any media image, localizes faces, processes geometry features, evaluates age, and displays results.

In [ ]:
def predict_child_age_from_image(image_np: np.ndarray) -> dict:
    """
    Locates face, extracts geometric features, runs classifier, and returns classification report.
    """
    # 1. Face detection & crop
    cropped_face, bbox = detect_and_crop_face(image_np)
    
    # 2. Feature extraction
    feats = extract_features(cropped_face, bbox)
    
    # 3. Model classification
    pred_class = mlp_model.predict([feats])[0]
    probabilities = mlp_model.predict_proba([feats])[0]
    prob_child = probabilities[0]
    
    classification = "Child" if pred_class == 0 else "Not a Child"
    
    # Plot original and cropped face bounding boxes
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(image_np)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    if bbox is not None:
        x, y, w, h = bbox
        rect = plt.Rectangle((x, y), w, h, fill=False, color='green', linewidth=2)
        axes[0].add_patch(rect)
        
    axes[1].imshow(cropped_face)
    axes[1].set_title(f"Cropped Face (128x128)\nRoundness Score: {feats[1]:.3f}")
    axes[1].axis('off')
    
    plt.suptitle(f"Age Category: {classification} ({prob_child:.2%} Child probability)", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return {
        "classification": classification,
        "child_probability": round(prob_child, 4),
        "features": {
            "aspect_ratio": feats[0],
            "roundness": feats[1]
        }
    }

# Run inference on a test synthetic child face
test_face = generate_synthetic_face(is_child=True)
results = predict_child_age_from_image(test_face)